# any-reduce-axis — worked example 2: Reduce the middle axis of a 3-D mask

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `any-reduce-axis`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`.any(dim=k)` works on tensors of any rank: it removes exactly the one axis you name and ORs over it. For a `(B, R, C)` tensor, `dim=1` collapses the R axis, leaving `(B, C)`. The remaining axes keep their order. This is the same convention as the 2-D case, just generalized.

## Worked solution

We have a `(batch, rays, segments)` boolean tensor `hits` where `hits[b, r, s]` is `True` if ray `r` in scene `b` intersects segment `s`. We want a `(batch, segments)` answer: for each scene and segment, does *any* ray hit that segment?

**Step 1 — name the axis that must vanish.** "Any ray" means OR across the ray axis. Rays are axis 1. We want axis 1 gone while batch (0) and segments (2) survive.

**Step 2 — apply `.any(dim=1)`.** Reducing dim=1 of `(B, R, C)` yields `(B, C)` = `(batch, segments)`. The OR runs independently for each `(b, s)` pair over all rays.

**Step 3 — verify shape and dtype.** Output rank drops by one (3 -> 2) because we did not pass `keepdim=True`. Dtype is `torch.bool`.

The general rule: the `dim` you pass is always the axis that disappears; every other axis stays in place.

In [ ]:
def segment_hit_by_any_ray(hits):
    # hits: (batch, rays, segments) bool -> (batch, segments) bool
    return hits.any(dim=1)

t.manual_seed(0)
hits = t.rand(2, 3, 4) > 0.6  # (batch=2, rays=3, segments=4)
result = segment_hit_by_any_ray(hits)
print(result, result.shape, result.dtype)